# 04 · xFG Success Modeling (M1 & M2)

**Purpose:** Scaffold cloglog success models with and without IPW/control-function adjustments.

**Inputs:** Reference/pbp_head.csv, reports/attempt_p_hat_sample.csv

**Outputs:** reports/xfg_success_metrics_placeholder.csv, reports/random_effects_stub.csv

**Sections:**
- [Parameters & Modes](#parameters--modes)
- [Imports](#imports--install-if-missing)
- [Utilities & Helpers](#utilities--helpers)
- [Data Load & Peek](#data-load--peek)
- [Stage Logic](#stage-logic)
- [Artifacts](#artifacts)
- [Session Info](#session-info)


In [ ]:

# Parameters & Modes

SMOKE_MODE <- TRUE
FULL_MODE <- !SMOKE_MODE

reference_dir <- 'Reference'
data_dir <- 'data'
reports_dir <- 'reports'
config_path <- file.path('config', 'params.yaml')

if (!dir.exists(reports_dir)) {
  dir.create(reports_dir, recursive = TRUE, showWarnings = FALSE)
}

default_params <- list(
  time_knots = c(60, 120, 300),
  late_flags = c(120, 60),
  p_clip_min = 0.05,
  p_clip_max = 0.98,
  kickable_cap_modeling = 65,
  kickable_cap_audit = 50,
  tau_grid = c(0.03, 0.05, 0.07, 0.10),
  df_distance = 5,
  df_yardline = 5,
  df_yards_to_go = 5
)

params <- default_params

if (file.exists(config_path)) {
  tryCatch({
    config_values <- yaml::read_yaml(config_path)
    params <- utils::modifyList(params, config_values, keep.null = TRUE)
  }, error = function(e) message('Config read failed, using defaults: ', e$message))
}

list2env(params, envir = .GlobalEnv)

set.seed(20240517)


In [ ]:

# Imports — install if missing

dependencies <- c(
  'dplyr', 'tibble', 'tidyr', 'readr', 'stringr', 'purrr', 'ggplot2',
  'mgcv', 'splines', 'glmmTMB', 'pROC', 'yaml', 'rlang'
)

installed <- rownames(installed.packages())

for (pkg in dependencies) {
  if (!pkg %in% installed) {
    install.packages(pkg)
  }
  suppressPackageStartupMessages(library(pkg, character.only = TRUE))
}


### Function Index

- `get_schema()` — quick schema preview of a data frame.
- `ensure_columns()` — add defaulted columns when missing.
- `add_time_features()` — derive common time and score features.
- `clip_probabilities()` — constrain probabilities to [min, max].
- `stabilize_weights()` — compute stabilized weights with optional grouping.
- `effective_sample_size()` — calculate ESS from weights.
- `calibration_summary()` — summarize calibration by bins.
- `plot_calibration()` — simple calibration scatter + smoother.


In [ ]:

# Utilities & Helpers (≤40 lines each)

get_schema <- function(df, n = 5) {
  tibble::tibble(
    name = names(df),
    class = purrr::map_chr(df, ~ paste(class(.x), collapse = '/')),
    sample = purrr::map_chr(df, ~ paste(head(.x, n), collapse = ', '))
  )
}

ensure_columns <- function(df, defaults) {
  for (col in names(defaults)) {
    if (!col %in% names(df)) {
      df[[col]] <- defaults[[col]]
    }
  }
  df
}

add_time_features <- function(df) {
  df %>%
    dplyr::mutate(
      time_remaining = dplyr::coalesce(game_seconds_remaining, 0),
      log_time_remaining = log1p(pmax(time_remaining, 0)),
      late_game_bucket = dplyr::case_when(
        time_remaining <= late_flags[2] ~ 'inside_1_minute',
        time_remaining <= late_flags[1] ~ 'two_minutes',
        TRUE ~ 'early'
      ),
      distance_bucket = cut(field_goal_distance, breaks = c(0, 30, 40, 50, 60, Inf), right = FALSE),
      wind_bucket = cut(weather_wind_mph, breaks = c(-Inf, 5, 10, 15, 25, Inf), right = FALSE)
    )
}

clip_probabilities <- function(x, lower = 0.05, upper = 0.95) {
  pmin(pmax(x, lower), upper)
}

stabilize_weights <- function(p_hat, groups = NULL, clip = c(0.05, 0.98)) {
  p_hat <- clip_probabilities(p_hat, clip[1], clip[2])
  if (is.null(groups)) {
    target <- mean(p_hat, na.rm = TRUE)
    weights <- target / p_hat
  } else {
    weights <- dplyr::tibble(group = groups, p_hat = p_hat) %>%
      dplyr::group_by(group) %>%
      dplyr::mutate(target = mean(p_hat, na.rm = TRUE)) %>%
      dplyr::ungroup() %>%
      dplyr::transmute(weight = target / p_hat) %>%
      dplyr::pull(weight)
  }
  clip_probabilities(weights, clip[1], 1 / clip[1])
}

effective_sample_size <- function(weights) {
  if (length(weights) == 0 || all(is.na(weights))) {
    return(NA_real_)
  }
  w <- weights[is.finite(weights)]
  if (length(w) == 0) {
    return(NA_real_)
  }
  sum_w <- sum(w)
  sum_sq <- sum(w^2)
  if (sum_sq == 0) {
    return(NA_real_)
  }
  sum_w^2 / sum_sq
}

calibration_summary <- function(df, truth_col, estimate_col, weight_col = NULL, by = NULL, bins = 5) {
  if (!is.null(by) && length(by) > 0) {
    df <- df %>%
      dplyr::mutate(dplyr::across(dplyr::all_of(by), as.factor))
  }
  if (!('calibration_bin' %in% names(df))) {
    df <- df %>%
      dplyr::mutate(calibration_bin = cut(.data[[estimate_col]], breaks = bins, include.lowest = TRUE))
  }
  df %>%
    dplyr::group_by(dplyr::across(dplyr::all_of(c(by, 'calibration_bin')))) %>%
    dplyr::summarise(
      truth = mean(.data[[truth_col]], na.rm = TRUE),
      estimate = mean(.data[[estimate_col]], na.rm = TRUE),
      weight = if (is.null(weight_col)) dplyr::n() else sum(.data[[weight_col]], na.rm = TRUE),
      .groups = 'drop'
    )
}

plot_calibration <- function(df, truth_col = 'truth', estimate_col = 'estimate') {
  ggplot2::ggplot(df, ggplot2::aes(x = .data[[estimate_col]], y = .data[[truth_col]])) +
    ggplot2::geom_point() +
    ggplot2::geom_abline(slope = 1, intercept = 0, linetype = 'dashed', colour = 'grey50') +
    ggplot2::labs(x = 'Estimated', y = 'Observed', title = 'Calibration Plot')
}


In [ ]:

# Data Load & Peek

pbp_path <- file.path(reference_dir, 'pbp_head.csv')
fg_path <- file.path(reference_dir, 'fg_attempts_sample.csv')

pbp <- readr::read_csv(pbp_path, show_col_types = FALSE)
fg_attempts <- if (file.exists(fg_path)) readr::read_csv(fg_path, show_col_types = FALSE) else NULL

pbp <- ensure_columns(pbp, list(
  kickable_fourth_down = FALSE,
  field_goal_distance = NA_real_,
  score_differential = 0,
  game_seconds_remaining = 0,
  yards_to_go = 10,
  season = 2015,
  coach = 'unknown',
  weather_temp = 65,
  weather_wind_mph = 0,
  roof = 'outdoor',
  precip = 'none',
  kicker = 'unknown',
  stadium = 'unknown',
  yardline = 50,
  play_type = 'field_goal',
  result = 'made',
  play_id = 0L,
  game_id = 'G-0000',
  play_description = 'FG'
))

if (!is.null(fg_attempts)) {
  fg_attempts <- ensure_columns(fg_attempts, list(
    play_id = 0L,
    game_id = 'G-0000',
    result = 'made'
  ))
}

if (SMOKE_MODE) {
  pbp <- pbp %>% dplyr::slice_head(n = min(1000, dplyr::n()))
  if (!is.null(fg_attempts)) {
    fg_attempts <- fg_attempts %>% dplyr::slice_head(n = min(500, dplyr::n()))
  }
}

glimpse(pbp)


In [ ]:

# Stage Logic — Success Modeling (M1 / M2)

## TODO: implement cloglog glmmTMB fits for kicks-only (M1) and IPW + control-function model (M2).

kicks <- pbp %>%
  dplyr::filter(play_type == 'field_goal') %>%
  add_time_features() %>%
  dplyr::mutate(
    outcome = dplyr::if_else(result == 'made', 1, 0),
    is_pat = dplyr::if_else(stringr::str_detect(play_description, 'PAT'), TRUE, FALSE, FALSE)
  )

attempt_weights <- dplyr::left_join(
  kicks,
  readr::read_csv(file.path(reports_dir, 'attempt_p_hat_sample.csv'), show_col_types = FALSE),
  by = c('game_id', 'play_id', 'season')
)

modeled <- attempt_weights %>%
  dplyr::mutate(
    pi_hat = dplyr::coalesce(pi_hat, 1),
    weight = dplyr::if_else(is_pat, 1, dplyr::coalesce(weight, 1)),
    control_function = pi_hat - mean(pi_hat, na.rm = TRUE)
  )

metrics <- tibble::tibble(
  model = c('M1_baseline', 'M2_ipw_control'),
  auc = NA_real_,
  brier = NA_real_,
  log_loss = NA_real_
)

readr::write_csv(metrics, file.path(reports_dir, 'xfg_success_metrics_placeholder.csv'))
readr::write_csv(
  tibble::tibble(effect = c('kicker', 'stadium'), note = 'pending model fit'),
  file.path(reports_dir, 'random_effects_stub.csv')
)

metrics


In [ ]:

# Artifacts

message('Success model placeholders written to reports/xfg_success_metrics_placeholder.csv')


### Optional Experiment — M3 Template

- Cross-fit xFG on kicks using the same rolling-origin splits as π.
- Score non-attempts with models that did not observe them to create pseudo labels.
- Combine kicks and pseudo-labeled non-attempts with down-weighted influence (λ ∈ {0.10, 0.20, 0.30}).
- Evaluate via AIPW calibration; keep the model only if low-overlap performance improves without harming kicks-only metrics.


In [ ]:

    # Session Info

    info <- capture.output(sessionInfo())
    readr::write_lines(info, file.path(reports_dir, 'session_info.txt'), append = TRUE)
    cat(info, sep = '
')
